<a href="https://colab.research.google.com/github/jrhumberto/2026-mestrado-pep/blob/main/unir_bases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Etapa Inicial: Instalação de Pacotes

In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


governadores = pd.read_csv('https://raw.githubusercontent.com/jrhumberto/000-pep/refs/heads/main/data/eleicoes_uf_2018_2022_austeridade.csv')
endividamento = pd.read_csv('https://raw.githubusercontent.com/jrhumberto/000-pep/refs/heads/main/data/endividamento_estados_2018_2024.csv')


endividamento['austeridade'] = 0



In [45]:
endividamento.head(50)

,uf,ano,dcl_valor,pessoal_pct_rcl,austeridade
0,AC,2018,3.565447e+09,48.01,0
1,AC,2019,3.116892e+09,53.74,0
2,AC,2020,3.337030e+09,52.69,0
3,AC,2021,2.847799e+09,51.44,0
4,AC,2022,2.505322e+09,46.36,0
5,AC,2023,2.051524e+09,48.41,0
6,AC,2024,1.978575e+09,46.77,0
7,AL,2018,6.816346e+09,48.69,0
8,AL,2019,6.404122e+09,44.71,0
9,AL,2020,5.813490e+09,39.78,0


In [46]:
import numpy as np
list_year = list(endividamento.ano.unique())
list_uf = list(endividamento.uf.unique())


for x in list_uf:
  for y in list_year:
    pleito = 2018 if y <=2022 else 2022 # Se o exercicio for 2019, 2020, 2021 e 2022 pegará a austeridade do candidato eleito em 2018;
    f1 = governadores['ANO_ELEICAO'] == z
    f2 = governadores['SG_UF'] == x
    austeridade = int(governadores[ f1 & f2]['austeridade'].iloc[0])
    endividamento.loc[ (endividamento['uf'] == x) & (endividamento['ano'] == y), 'austeridade'] = austeridade


endividamento.head(50)

,uf,ano,dcl_valor,pessoal_pct_rcl,austeridade
0,AC,2018,3.565447e+09,48.01,0
1,AC,2019,3.116892e+09,53.74,0
2,AC,2020,3.337030e+09,52.69,0
3,AC,2021,2.847799e+09,51.44,0
4,AC,2022,2.505322e+09,46.36,0
5,AC,2023,2.051524e+09,48.41,0
6,AC,2024,1.978575e+09,46.77,0
7,AL,2018,6.816346e+09,48.69,0
8,AL,2019,6.404122e+09,44.71,0
9,AL,2020,5.813490e+09,39.78,0


In [47]:
from scipy import stats

df = endividamento.copy()

# Separar grupos
s0 = df.loc[df['austeridade'] == 0, 'dcl_valor']
s1 = df.loc[df['austeridade'] == 1, 'dcl_valor']
# Teste t com variâncias desiguais (Welch)
t_stat, p_val = stats.ttest_ind(
s1, s0,
equal_var=False,
nan_policy='omit'
)
print(f"Teste t (Welch) dcl_valor - austeridade vs outros")
print(f"t = {t_stat:.3f}, p-valor = {p_val:.4f}")
print(f"Média grupo 0 (não austeridade): {s0.mean():.2f}")
print(f"Média grupo 1 (austeridade): {s1.mean():.2f}")

Teste t (Welch) dcl_valor - austeridade vs outros
t = 5.729, p-valor = 0.0000
Média grupo 0 (não austeridade): 5332081714.27
Média grupo 1 (austeridade): 53697496310.98


In [50]:
import statsmodels.formula.api as smf
# Criar dummies de UF e ano (efeitos fixos)
uf_d = pd.get_dummies(df['uf'], prefix='uf', drop_first=True)
ano_d = pd.get_dummies(df['ano'], prefix='ano', drop_first=True)
model_df = pd.concat([df, uf_d, ano_d], axis=1)

fe_uf = ' + '.join(uf_d.columns)
fe_ano = ' + '.join(ano_d.columns)

formula = (
'austeridade ~ dcl_valor +  pessoal_pct_rcl +' + fe_uf + ' + ' + fe_ano
)
model = smf.ols(formula=formula, data=model_df).fit(cov_type='HC3') # erros robustos
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            austeridade   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 1.357e+13
Date:                Tue, 24 Feb 2026   Prob (F-statistic):               0.00
Time:                        07:42:25   Log-Likelihood:                 2025.5
No. Observations:                 189   AIC:                            -3981.
Df Residuals:                     154   BIC:                            -3868.
Df Model:                          34                                         
Covariance Type:                  HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept         6.661e-16    1.1e-05  

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 34, but rank is 33
  warnings.warn('covariance of constraints does not have full '
